In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

landing_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/shipments/"
)

bronze_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/shipments/"
)

checkpoint_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "checkpoints/bronze_shipments/"
)

schema_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "checkpoints/schemas/bronze_shipments/"
)

# ============================================================
# 2. SCHÉMA SOURCE
# ============================================================

shipments_schema = StructType([
    StructField("shipment_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("carrier_id", StringType(), True),
    StructField("shipment_time", TimestampType(), True),
    StructField("expected_delivery_time", TimestampType(), True),
    StructField("actual_delivery_time", TimestampType(), True),
    StructField("origin_city", StringType(), True),
    StructField("destination_city", StringType(), True),
    StructField("distance_km", DecimalType(10, 2), True),
    StructField("shipment_status", StringType(), True),
    StructField("updated_at", TimestampType(), True)
])

# ============================================================
# 3. LECTURE INCRÉMENTALE AVEC AUTO LOADER
# ============================================================

shipments_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_shipments_path)
    .option("header", "true")
    .schema(shipments_schema)
    .load(landing_shipments_path)
)

# ============================================================
# 4. AJOUT DES MÉTADONNÉES BRONZE
# ============================================================

bronze_shipments_df = (
    shipments_stream
    .withColumn(
        "source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "batch_id",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"(batch_\d+)",
            1
        )
    )
)

# ============================================================
# 5. ÉCRITURE DELTA
# ============================================================

query = (
    bronze_shipments_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_shipments_path
    )
    .option(
        "path",
        bronze_shipments_path
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("Ingestion Bronze shipments terminée.")

# ============================================================
# 6. LECTURE DE LA BRONZE
# ============================================================

bronze_shipments = (
    spark.read
    .format("delta")
    .load(bronze_shipments_path)
)

print(
    "Nombre de lignes Bronze shipments :",
    bronze_shipments.count()
)

display(
    bronze_shipments
    .orderBy("shipment_id")
)

# ============================================================
# 7. VÉRIFICATION DES MÉTADONNÉES
# ============================================================

display(
    bronze_shipments.select(
        "shipment_id",
        "order_id",
        "carrier_id",
        "source_file",
        "batch_id",
        "ingestion_timestamp"
    )
    .orderBy("shipment_id")
)